In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from selenium import webdriver
import time
from collections import OrderedDict


user_link = input("URL: ")

# start web browser
browser=webdriver.Chrome()

# get source code
browser.get(user_link)
html = browser.page_source
time.sleep(5)
print(html)

# close web browser
browser.close()


#content = requests.get(url).content 
soup = BeautifulSoup(html, "html.parser")



#get all classes names from html
classes = [value
           for element in soup.find_all(class_=True)
           for value in element["class"]]


#remove all redundant classes
print(len(classes))
classes_nd= set(classes)
classes= list(classes_nd)
print(len(classes_nd))



#get all text for each class name
classes_list=[]
for i in range(len(classes)):
    class_data = [x.get_text() for x in soup.find_all(class_=classes[i])]
    class_data = list(map(lambda x:x.replace("\n",""),class_data)) #remove all (newline)
    class_data = list(map(lambda x:x.replace("  ",""),class_data)) #remove all (white spaces)
    classes_list.append(class_data)


In [ ]:
#convert list to dictionary
data_dict={}

for i in range(len(classes)):
    data_dict.update({classes[i]:classes_list[i]})


    
#remove all equal and empty columns    
def all_equal(iterator):
    iterator = iter(iterator)
    try:
        first = next(iterator)
    except StopIteration:
        return True
    return all(first == x for x in iterator)
       
for key, value in list(data_dict.items()):
    if all_equal(value) == True:
        del data_dict[key]


#Sort dictionary based on count
data_dict = OrderedDict(sorted(data_dict.items(), key=lambda x: len(x[1])))
data_dict = OrderedDict(reversed(list(data_dict.items())))

#remove all keys that less then (col_num)
col_num = 1
data_dict = dict((k,v) for k,v in data_dict.items() if len(v) > col_num)        
    
    
#print dictionary
kj = {k: data_dict[k] for k in list(data_dict)[:10]}    
print(kj)

In [ ]:
#convert dictionary to DataFrame 
df = pd.DataFrame.from_dict(data_dict, orient='index')
df = df.transpose()

#save DataFrame as .csv file

df.to_csv('test.csv', index=False, encoding='utf-8')
# df.to_excel('test.xlsx', index=False, encoding='utf-8')
# df.to_json('test.json', orient='records')
data = pd.read_csv('test.csv')
